# SinLlama Classification-Head Training

Train only the sequence-classification head for `polyglots/SinLlama_v01`. The SinLlama backbone is frozen, and the benchmark still trains on one selected condition and evaluates on clean, transliteration-heavy, and codemix-heavy test sets.


## 1. Install dependencies

Run once, then restart the runtime if packages were newly installed.

In [ ]:
%pip install -q -U "transformers>=4.45" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.46.1" peft pandas scikit-learn matplotlib seaborn huggingface_hub safetensors


## 2. Configuration

Select one training condition with `DATASET_CHOICE`. This notebook freezes the SLM backbone and trains only the classification head.


In [ ]:
from pathlib import Path
import dataclasses
import gc
import inspect
import json
import os
import time

import numpy as np
import pandas as pd
import torch

SEED = 42
NUM_LABELS = 2
LABEL_NAME = {0: "NOT", 1: "OFF"}
LABEL2ID = {v: k for k, v in LABEL_NAME.items()}

# Options include: "clean", "codemix_heavy", "transliteration_heavy", "compound_heavy", "misspell_heavy".
# Test-only variants are also available for evaluation: "compound_light", "misspell_light".
DATASET_CHOICE = "clean"
TEST_DATASETS = ["clean", "codemix_heavy", "transliteration_heavy", "compound_heavy", "misspell_heavy"]
TRAIN_DATASETS = [DATASET_CHOICE]
TRAIN_SUBSET = None  # e.g. 200 for a smoke-test slice

DATA_ROOT = None
MODEL_ID = "polyglots/SinLlama_v01"
MODEL_NAME = "sinllama_classification_head"
RUN_NAME = f"sinllama_head_only_{DATASET_CHOICE}"
OUTPUT_ROOT = Path("runs") / RUN_NAME

EPOCHS = 3.0
LEARNING_RATE = 1e-4  # Higher learning rate for head-only training
TRAIN_BATCH_SIZE = 8  # Base model is frozen, so this can usually be larger
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 32
MAX_LENGTH = 128
VALIDATION_FRACTION = 0.10
LOGGING_STEPS = 10
OPTIMIZER = "adamw_torch"

np.random.seed(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"cuda:{i} {props.name}, {props.total_memory / 2**30:.1f} GiB")
else:
    print("CUDA is not available. This notebook can run on CPU for debugging, but SinLlama training will be very slow.")


## 3. Authenticate and locate data

No token is stored in this notebook. Set `HF_TOKEN` as an environment/Kaggle secret if the SinLlama repo requires authentication in your environment.


In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated from HF_TOKEN.")
else:
    print("HF_TOKEN is not set. Continuing with normal Hugging Face access.")

DATASETS = {
    "clean": {"train": Path("sold/train.jsonl"), "test": Path("sold/test.jsonl")},
    "codemix_light": {"train": Path("outputs/train/codemix_light.jsonl"), "test": Path("outputs/test/codemix_light.jsonl")},
    "codemix_heavy": {"train": Path("outputs/train/codemix_heavy.jsonl"), "test": Path("outputs/test/codemix_heavy.jsonl")},
    "transliteration_light": {"train": Path("outputs/train/transliteration_light.jsonl"), "test": Path("outputs/test/transliteration_light.jsonl")},
    "transliteration_heavy": {"train": Path("outputs/train/transliteration_heavy.jsonl"), "test": Path("outputs/test/transliteration_heavy.jsonl")},
    "compound_light": {"test": Path("outputs/test/compound_light.jsonl")},
    "compound_heavy": {"train": Path("outputs/train/compound_heavy.jsonl"), "test": Path("outputs/test/compound_heavy.jsonl")},
    "misspell_light": {"test": Path("outputs/test/misspell_light.jsonl")},
    "misspell_heavy": {"train": Path("outputs/train/misspell_heavy.jsonl"), "test": Path("outputs/test/misspell_heavy.jsonl")},
}

DATASET_ALIASES = {
    "codemix": "codemix_heavy",
    "transliteration": "transliteration_heavy",
    "compound": "compound_heavy",
    "misspell": "misspell_heavy",
}

TRAIN_DATASETS = [DATASET_ALIASES.get(name, name) for name in TRAIN_DATASETS]
TEST_DATASETS = [DATASET_ALIASES.get(name, name) for name in TEST_DATASETS]

unknown_datasets = (set(TRAIN_DATASETS) | set(TEST_DATASETS)) - set(DATASETS)
if unknown_datasets:
    raise KeyError(f"Undefined dataset names: {sorted(unknown_datasets)}")
if not TRAIN_DATASETS or not TEST_DATASETS:
    raise ValueError("TRAIN_DATASETS and TEST_DATASETS must both be non-empty.")

required_paths = []
missing_train_paths = [name for name in TRAIN_DATASETS if "train" not in DATASETS[name]]
missing_test_paths = [name for name in TEST_DATASETS if "test" not in DATASETS[name]]
if missing_train_paths:
    raise KeyError(f"No training file is configured for: {missing_train_paths}")
if missing_test_paths:
    raise KeyError(f"No test file is configured for: {missing_test_paths}")

for name in TRAIN_DATASETS:
    required_paths.append(DATASETS[name]["train"])
for name in TEST_DATASETS:
    required_paths.append(DATASETS[name]["test"])

def is_data_root(path):
    path = Path(path)
    return all((path / rel).is_file() for rel in required_paths)

def find_data_root(explicit=None):
    if explicit is not None:
        explicit = Path(explicit).expanduser()
        if not is_data_root(explicit):
            raise FileNotFoundError(f"Selected files were not found below DATA_ROOT: {explicit.resolve()}")
        return explicit
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent / "data", parent):
            if is_data_root(candidate):
                return candidate
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        for candidate in kaggle.iterdir():
            if candidate.is_dir() and is_data_root(candidate):
                return candidate
    raise FileNotFoundError("Could not locate all selected files. Set DATA_ROOT explicitly.")

DATA_ROOT = find_data_root(DATA_ROOT)
FILES = {}
for name in TRAIN_DATASETS:
    FILES[f"{name}_train"] = DATA_ROOT / DATASETS[name]["train"]
for name in TEST_DATASETS:
    FILES[f"{name}_test"] = DATA_ROOT / DATASETS[name]["test"]

print("DATA_ROOT =", DATA_ROOT.resolve())
print("MODEL_ID =", MODEL_ID)
display(pd.DataFrame([{"key": key, "path": str(path)} for key, path in FILES.items()]))


## 4. Load and validate datasets

This cell accepts the clean SOLD schema, generated noisy schemas, and token-list JSONL records, then canonicalizes them to `id`, `text`, and `label`.


In [ ]:
def read_jsonl(path):
    records = []
    with Path(path).open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path}, line {line_number}: {exc}") from exc
    if not records:
        raise ValueError(f"Dataset is empty: {path}")
    return pd.DataFrame(records)

def normalize_label_series(labels):
    mapping = {
        "0": 0, "1": 1,
        "NOT": 0, "OFF": 1,
        "not": 0, "off": 1,
        "negative": 0, "positive": 1,
        "not-offensive": 0, "offensive": 1,
        "non-hate": 0, "hate": 1,
    }
    normalized = labels.astype(str).str.strip().map(mapping)
    if normalized.isna().any():
        bad = sorted(labels[normalized.isna()].astype(str).unique().tolist())
        raise ValueError(f"Unmapped labels: {bad}")
    return normalized.astype(int)

def canonicalize(df, condition, split):
    id_candidates = ("post_id", "source_id", "id", "sentence_id", "orig_id")
    text_candidates = ("text", "noisy_text", "transliterated", "transliteration", "romanized")
    id_col = next((col for col in id_candidates if col in df.columns), None)
    text_col = next((col for col in text_candidates if col in df.columns), None)
    if text_col is None and "tokens" in df.columns:
        df = df.copy()
        df["text"] = df["tokens"].apply(lambda value: " ".join(value) if isinstance(value, list) else str(value))
        text_col = "text"
    if id_col is None:
        df = df.copy()
        df["id"] = [f"{condition}_{split}_{i}" for i in range(len(df))]
        id_col = "id"
    if text_col is None or "label" not in df.columns:
        raise KeyError(f"{condition}_{split}: unsupported columns {df.columns.tolist()}")
    out = df[[id_col, text_col, "label"]].rename(columns={id_col: "id", text_col: "text"}).copy()
    out["id"] = out["id"].astype(str)
    out["text"] = out["text"].astype(str)
    out["label"] = normalize_label_series(out["label"])
    if out["id"].duplicated().any():
        out["id"] = [f"{condition}_{split}_{i}" for i in range(len(out))]
    if out["text"].str.strip().eq("").any():
        raise ValueError(f"{condition}_{split}: empty text found")
    out["condition"] = condition
    out["split"] = split
    return out

frames = {}
for key, path in FILES.items():
    condition, split = key.rsplit("_", 1)
    frames[key] = canonicalize(read_jsonl(path), condition, split)
    counts = frames[key]["label"].value_counts().sort_index().to_dict()
    print(f"{key:32s} n={len(frames[key]):5d} labels={counts}")

for split, names in (("train", TRAIN_DATASETS), ("test", TEST_DATASETS)):
    split_frames = {name: frames[f"{name}_{split}"].set_index("id") for name in names}
    for left_index, left_name in enumerate(names):
        for right_name in names[left_index + 1:]:
            left, right = split_frames[left_name], split_frames[right_name]
            if set(left.index) == set(right.index):
                right = right.loc[left.index]
                if not np.array_equal(left["label"].to_numpy(), right["label"].to_numpy()):
                    raise ValueError(f"Label mismatch between aligned {left_name} and {right_name} {split} data")

preview = []
for name in TEST_DATASETS:
    row = frames[f"{name}_test"].head(2).copy()
    row["dataset"] = name
    preview.append(row[["dataset", "id", "text", "label"]])
display(pd.concat(preview, ignore_index=True).assign(label_name=lambda df: df["label"].map(LABEL_NAME)))


## 5. Tokenizer, frozen SLM, and datasets

The SLM is loaded as a sequence-classification model. All backbone parameters are frozen, and only the classification head is left trainable.


In [ ]:
import torch.nn as nn
from accelerate import init_empty_weights
from datasets import ClassLabel, Dataset
from huggingface_hub import hf_hub_download, list_repo_files
from peft import PeftModel
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    set_seed,
)
from transformers.models.llama.modeling_llama import LlamaForSequenceClassification

BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
DEVICE_MAP = "auto" if torch.cuda.is_available() else None
LOAD_IN_4BIT = True


def quantization_config():
    if not LOAD_IN_4BIT:
        return None
    if not torch.cuda.is_available():
        print("4-bit quantization requires CUDA; loading without quantization for CPU debugging.")
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )


def build_tokenizer(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=True, token=hf_token)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print(f"pad_token -> {tokenizer.pad_token!r}")
    print(f"vocab={len(tokenizer)} pad_id={tokenizer.pad_token_id} padding_side={tokenizer.padding_side}")
    probe = "මම පාසල් යනවා. ඔයා ගොඩක් මෝඩයි නේද"
    fertility = len(tokenizer.tokenize(probe)) / max(len(probe.split()), 1)
    print(f"Sinhala token fertility sanity check: {fertility:.2f} tokens/word")
    return tokenizer


def make_dataset(frame, tokenizer):
    dataset = Dataset.from_pandas(frame[["id", "text", "label"]], preserve_index=False)
    dataset = dataset.cast_column("label", ClassLabel(names=["NOT", "OFF"]))
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )


def set_trainable_classification_head(model):
    for parameter in model.parameters():
        parameter.requires_grad = False

    head_markers = ("score", "classifier", "classification_head")
    trainable_names = []
    for name, parameter in model.named_parameters():
        if any(marker in name for marker in head_markers):
            parameter.requires_grad = True
            trainable_names.append(name)
            print("TRAINABLE:", name, tuple(parameter.shape))

    if not trainable_names:
        print("No classification head was found. Parameters containing likely head names:")
        for name, parameter in model.named_parameters():
            if any(marker in name.lower() for marker in head_markers):
                print(name, tuple(parameter.shape))
        raise RuntimeError("Could not find a trainable classification head. Inspect model.named_parameters().")

    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.6f}%)")
    return trainable_names, total, trainable


def adapter_info(model_id):
    try:
        files = list_repo_files(model_id, token=hf_token)
    except Exception as error:
        print(f"repo listing failed ({error}); assuming merged/full model weights")
        return False, model_id, []
    if "adapter_config.json" not in files:
        return False, model_id, []

    adapter_config_path = hf_hub_download(model_id, "adapter_config.json", token=hf_token)
    with open(adapter_config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    base_model_id = config.get("base_model_name_or_path", "meta-llama/Meta-Llama-3-8B")
    saved_modules = config.get("modules_to_save") or []
    print(f"{model_id} is a PEFT adapter over {base_model_id}; modules_to_save={saved_modules}")
    return True, base_model_id, saved_modules


def sequence_classification_config(base_model_id, tokenizer):
    config = AutoConfig.from_pretrained(base_model_id, token=hf_token, trust_remote_code=True)
    config.vocab_size = len(tokenizer)
    config.num_labels = NUM_LABELS
    config.id2label = LABEL_NAME
    config.label2id = LABEL2ID
    config.pad_token_id = tokenizer.pad_token_id
    config.use_cache = False
    return config


def build_model(model_id, tokenizer):
    is_adapter, base_model_id, saved_modules = adapter_info(model_id)
    qconfig = quantization_config()
    common_kwargs = dict(
        torch_dtype=COMPUTE_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
        token=hf_token,
        low_cpu_mem_usage=True,
    )
    if qconfig is not None:
        common_kwargs["quantization_config"] = qconfig
        print("Loading base model in 4-bit NF4 quantization.")


    if not is_adapter:
        model = AutoModelForSequenceClassification.from_pretrained(
            base_model_id,
            config=sequence_classification_config(base_model_id, tokenizer),
            ignore_mismatched_sizes=True,
            **common_kwargs,
        )
        if model.get_input_embeddings().weight.shape[0] != len(tokenizer):
            print(f"resizing token embeddings {model.get_input_embeddings().weight.shape[0]} -> {len(tokenizer)}")
            model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
    else:
        causal = AutoModelForCausalLM.from_pretrained(base_model_id, **common_kwargs)
        if causal.get_input_embeddings().weight.shape[0] != len(tokenizer):
            print(f"resizing token embeddings {causal.get_input_embeddings().weight.shape[0]} -> {len(tokenizer)}")
            causal.resize_token_embeddings(len(tokenizer), mean_resizing=False)

        causal = PeftModel.from_pretrained(causal, model_id, is_trainable=False, token=hf_token)
        causal = causal.merge_and_unload()
        print(f"merged SinLlama adapter into base model (modules_to_save={saved_modules})")

        config = sequence_classification_config(base_model_id, tokenizer)
        with init_empty_weights():
            model = LlamaForSequenceClassification(config)
        model.model = causal.model
        head_device = causal.model.norm.weight.device
        model.score = nn.Linear(config.hidden_size, NUM_LABELS, bias=False).to(device=head_device, dtype=COMPUTE_DTYPE)
        nn.init.normal_(model.score.weight, std=0.02)

        if hasattr(causal, "hf_device_map"):
            model.hf_device_map = causal.hf_device_map
        causal.lm_head = None
        del causal
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False

    head_names, total_params, trainable_params = set_trainable_classification_head(model)
    return model, {
        "fine_tuning_method": "classification_head_only",
        "model_load_type": "peft_adapter_merged" if is_adapter else "sequence_classification_model",
        "base_model_id": base_model_id,
        "load_in_4bit": qconfig is not None,
        "quantization_type": "nf4" if qconfig is not None else None,
        "head_parameter_names": head_names,
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
    }


## 6. Metrics and trainer

Macro-F1 is the checkpoint-selection metric. The evaluation outputs from the original notebook are preserved.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import Trainer, TrainingArguments


RESULT_COLUMNS = [
    "Model",
    "Trained Noise Condition",
    "Test Noise Condition",
    "Train Size",
    "Test Size",
    "Accuracy",
    "Macro F1",
    "balanced_accuracy",
    "Precision (Offensive)",
    "Recall (Offensive)",
    "F1 (Offensive)",
    "Precision (Not-Offensive)",
    "Recall (Not-Offensive)",
    "F1 (Not-Offensive)",
    "Support (Offensive)",
    "Support (Not-Offensive)",
]


def format_result_row(model_name, train_condition, test_condition, train_size, test_size, metrics):
    return {
        "Model": model_name,
        "Trained Noise Condition": train_condition,
        "Test Noise Condition": test_condition,
        "Train Size": train_size,
        "Test Size": test_size,
        "Accuracy": metrics["accuracy"],
        "Macro F1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "Precision (Offensive)": metrics["off_precision"],
        "Recall (Offensive)": metrics["off_recall"],
        "F1 (Offensive)": metrics["off_f1"],
        "Precision (Not-Offensive)": metrics["not_precision"],
        "Recall (Not-Offensive)": metrics["not_recall"],
        "F1 (Not-Offensive)": metrics["not_f1"],
        "Support (Offensive)": metrics["off_support"],
        "Support (Not-Offensive)": metrics["not_support"],
    }


def metric_values(labels, predictions):
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1], zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "not_precision": precision[0],
        "not_recall": recall[0],
        "not_f1": f1[0],
        "not_support": int(support[0]),
        "off_precision": precision[1],
        "off_recall": recall[1],
        "off_f1": f1[1],
        "off_support": int(support[1]),
    }


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    return metric_values(labels, np.argmax(logits, axis=-1))


def trainer_tokenizer_kwargs(tokenizer):
    parameters = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in parameters:
        return {"processing_class": tokenizer}
    if "tokenizer" in parameters:
        return {"tokenizer": tokenizer}
    return {}


def training_arguments(output_dir):
    kwargs = dict(
        output_dir=str(output_dir / "checkpoints"),
        seed=SEED,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        eval_accumulation_steps=8,
        learning_rate=LEARNING_RATE,
        fp16=COMPUTE_DTYPE == torch.float16 and torch.cuda.is_available(),
        bf16=COMPUTE_DTYPE == torch.bfloat16 and torch.cuda.is_available(),
        optim=OPTIMIZER,
        logging_steps=LOGGING_STEPS,
        eval_strategy="epoch",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        dataloader_num_workers=2,
        dataloader_pin_memory=False,
        group_by_length=True,
        label_names=["labels"],
    )
    valid = {field.name for field in dataclasses.fields(TrainingArguments)}
    return TrainingArguments(**{key: value for key, value in kwargs.items() if key in valid})


## 7. Run one classification-head training-condition experiment


In [ ]:
def evaluate_dataset(trainer, dataset, frame, train_condition, test_condition, output_dir):
    prediction_output = trainer.predict(dataset)
    predictions = np.argmax(prediction_output.predictions, axis=-1)
    labels = prediction_output.label_ids
    metrics = metric_values(labels, predictions)
    report = classification_report(
        labels,
        predictions,
        labels=[0, 1],
        target_names=[LABEL_NAME[0], LABEL_NAME[1]],
        zero_division=0,
        output_dict=True,
    )
    result = format_result_row(
        MODEL_NAME,
        train_condition,
        test_condition,
        None,
        len(dataset),
        metrics,
    )
    predictions_frame = pd.DataFrame({
        "id": frame["id"].to_numpy(),
        "model": MODEL_NAME,
        "train_dataset": train_condition,
        "test_dataset": test_condition,
        "label": labels,
        "prediction": predictions,
        "text": frame["text"].to_numpy(),
    })
    with (output_dir / f"{test_condition}_classification_report.json").open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2)
    return result, predictions_frame


def run_experiment(train_condition):
    print("\n" + "=" * 80)
    print(f"MODEL: {MODEL_NAME} | TRAIN: {train_condition} | METHOD: classification head only")
    print("=" * 80)

    set_seed(SEED)
    output_dir = OUTPUT_ROOT / MODEL_NAME / f"train_{train_condition}_seed{SEED}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_frame = frames[f"{train_condition}_train"].copy()
    if TRAIN_SUBSET and TRAIN_SUBSET > 0:
        train_frame = train_frame.sample(n=min(TRAIN_SUBSET, len(train_frame)), random_state=SEED).reset_index(drop=True)

    tokenizer = build_tokenizer(MODEL_ID)
    full_train = make_dataset(train_frame, tokenizer)
    split = full_train.train_test_split(
        test_size=VALIDATION_FRACTION,
        seed=SEED,
        stratify_by_column="label",
    )
    train_ds, validation_ds = split["train"], split["test"]

    model, parameter_info = build_model(MODEL_ID, tokenizer)
    trainer = Trainer(
        model=model,
        args=training_arguments(output_dir),
        train_dataset=train_ds,
        eval_dataset=validation_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        **trainer_tokenizer_kwargs(tokenizer),
    )

    started = time.time()
    trainer.train()
    trainer.save_model(output_dir / "model")
    tokenizer.save_pretrained(output_dir / "model")

    result_rows = []
    prediction_frames = []
    for test_condition in TEST_DATASETS:
        test_frame = frames[f"{test_condition}_test"].copy()
        test_ds = make_dataset(test_frame, tokenizer)
        result, predictions = evaluate_dataset(
            trainer, test_ds, test_frame, train_condition, test_condition, output_dir
        )
        result["Train Size"] = len(train_ds)
        result_rows.append(result)
        prediction_frames.append(predictions)
        print(f"{test_condition:24s} macro_f1={result['Macro F1']:.4f} accuracy={result['Accuracy']:.4f}")

    results = pd.DataFrame(result_rows, columns=RESULT_COLUMNS)
    predictions = pd.concat(prediction_frames, ignore_index=True)
    results.to_csv(output_dir / "metrics.csv", index=False)
    predictions.to_json(output_dir / "predictions.jsonl", orient="records", lines=True, force_ascii=False)
    with (output_dir / "run_summary.json").open("w", encoding="utf-8") as handle:
        json.dump({
            "model": MODEL_NAME,
            "model_id": MODEL_ID,
            "train_condition": train_condition,
            "test_conditions": TEST_DATASETS,
            "seed": SEED,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
            "max_length": MAX_LENGTH,
            "optimizer": OPTIMIZER,
            **parameter_info,
        }, handle, indent=2)

    del trainer, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results, predictions


## 8. Run configured benchmark

Start with `TRAIN_SUBSET=200` for a smoke test. Full results are written under `runs/sinllama_head_only_<dataset_choice>` by default.


In [ ]:
all_results = []
all_predictions = []
started = time.time()

for train_condition in TRAIN_DATASETS:
    results, predictions = run_experiment(train_condition)
    all_results.append(results)
    all_predictions.append(predictions)

    partial_results = pd.concat(all_results, ignore_index=True)
    partial_results.to_csv(OUTPUT_ROOT / "benchmark_summary_partial.csv", index=False)

results = pd.concat(all_results, ignore_index=True)
predictions = pd.concat(all_predictions, ignore_index=True)
results.to_csv(OUTPUT_ROOT / "benchmark_summary.csv", index=False)
predictions.to_json(OUTPUT_ROOT / "all_predictions.jsonl", orient="records", lines=True, force_ascii=False)

matrix = results.pivot_table(index=["Model", "Trained Noise Condition"], columns="Test Noise Condition", values="Macro F1")
matrix = matrix.reindex(columns=TEST_DATASETS)
matrix.to_csv(OUTPUT_ROOT / "macro_f1_matrix.csv")
display(results.round(4))
display(matrix.round(4))
print(f"Finished in {(time.time() - started) / 3600:.2f} hours")
print("Artifacts:", OUTPUT_ROOT.resolve())

## 9. Plot macro-F1 matrix


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_matrix = matrix.reindex(columns=TEST_DATASETS)
plt.figure(figsize=(8, max(3, 0.45 * len(plot_matrix))))
sns.heatmap(plot_matrix, annot=True, fmt=".4f", cmap="Blues", vmin=0, vmax=1)
plt.title("SinLlama classification-head-only macro-F1")
plt.xlabel("Test condition")
plt.ylabel("Training condition")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "macro_f1_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


## Reporting checklist

Report the SinLlama model ID/revision, trainable classification-head parameter names, trainable parameter count, training condition and size, validation policy, seed, epochs, learning rate, effective batch size, truncation length, and clean/codemix/transliteration metrics. Keep smoke-test results separate from full-data results.
